In [1]:
# 03_utility_and_risk.ipynb -- Compute a utility-retention proxy and reconstruction risk per masking tier, using a REAL LLM adversary when an API key is present (else a documented parametric fallback).
# The adversary tries to recover masked code identifiers from context; recovery rate = reconstruction risk.
# OpenAI config is read from a .env file placed in the repository root (one level above notebooks/). The API key is never printed.
# Three model tiers (weak/mid/strong) act as the adversary-strength sweep. Output paths are intentionally not printed (anonymity).

import os
import json
import hashlib
import urllib.request
import urllib.error
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Reproducibility & grayscale publication style -------------------------
SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "savefig.dpi": 600, "figure.dpi": 120, "font.size": 11,
    "axes.edgecolor": "0.2", "grid.color": "0.85", "image.cmap": "Greys",
})
GRAYS = ["0.10", "0.30", "0.50", "0.68", "0.82"]
MARKERS = ["o", "s", "^", "D", "v"]

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA = ROOT / "data"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
CACHE = DATA / "_adv_cache"
for d in (FIG, TAB, CACHE):
    d.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)

TIERS = ["C0", "C1", "C2", "C3", "C4"]
REVERSIBLE = {"C0": True, "C1": True, "C2": True, "C3": True, "C4": False}

with open(DATA / "masked_corpus.json", encoding="utf-8") as f:
    masked = json.load(f)
with open(DATA / "id_maps.json", encoding="utf-8") as f:
    id_maps = json.load(f)

# --- Utility proxy: retained FUNCTIONAL structure (grounded in masking counts) ---
total_ident = sum(r["identifier"] for r in masked["C2"]) or 1
total_lit = sum(r["literal"] for r in masked["C3"]) or 1
def masked_fracs(tier):
    return (sum(r["identifier"] for r in masked[tier]) / total_ident,
            sum(r["literal"] for r in masked[tier]) / total_lit)
W_FLOOR, W_U_ID, W_U_LIT = 0.30, 0.45, 0.25
def utility_retention(tier):
    id_m, lit_m = masked_fracs(tier)
    return W_FLOOR + W_U_ID * (1 - id_m) + W_U_LIT * (1 - lit_m)

util_rows = []
for tier in TIERS:
    ret = utility_retention(tier)
    util_rows.append({"tier": tier, "context_retention": ret,
                      "task_success_proxy": 1.0 / (1.0 + np.exp(-6.0 * (ret - 0.45)))})
util_df = pd.DataFrame(util_rows)
util_df.to_csv(TAB / "utility_by_tier.csv", index=False)

# --- Read OpenAI config from repo-root .env (key never printed) ------------
def load_env(path):
    cfg = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            cfg[k.strip()] = v.strip().strip('"').strip("'")
    return cfg

ENV = load_env(ROOT / ".env")
API_KEY = ENV.get("OPENAI_API_KEY", os.environ.get("OPENAI_API_KEY", ""))
USE_LLM = bool(API_KEY) and API_KEY.lower() != "none"
MODELS = {"weak": ENV.get("LLM_MODEL_WEAK", "gpt-4o-mini"),
          "mid": ENV.get("LLM_MODEL_MID", "gpt-5.4-mini"),
          "strong": ENV.get("LLM_MODEL_STRONG", "gpt-5.4")}

N_ADV_FILES = 50   # files sampled for the LLM adversary (larger sample tightens the CIs)
K_IDENTS = 12      # identifiers probed per file
BOOT = 2000        # bootstrap resamples for 95% confidence intervals
_rng = np.random.default_rng(SEED)

def boot_ci(vals):
    # Percentile bootstrap 95% CI for the mean of per-file recovery rates.
    if len(vals) == 0:
        return float("nan"), float("nan")
    arr = np.asarray(vals, dtype=float)
    means = arr[_rng.integers(0, len(arr), size=(BOOT, len(arr)))].mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

# --- Real LLM adversary (OpenAI Chat Completions) with on-disk caching ------
def _openai_chat(model, prompt):
    payload = {"model": model, "messages": [{"role": "user", "content": prompt}], "temperature": 0}
    def _post(body):
        req = urllib.request.Request("https://api.openai.com/v1/chat/completions",
                                     data=json.dumps(body).encode(),
                                     headers={"Authorization": f"Bearer {API_KEY}",
                                              "Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=90) as r:
            return json.loads(r.read())
    try:
        data = _post(payload)
    except urllib.error.HTTPError as e:
        msg = e.read().decode(errors="ignore")
        if e.code == 400 and "temperature" in msg:  # some newer models reject non-default temperature
            payload.pop("temperature")
            data = _post(payload)
        else:
            raise
    return data["choices"][0]["message"]["content"]

def _cached_chat(model, prompt):
    key = hashlib.sha256((model + "::" + prompt).encode()).hexdigest()
    cf = CACHE / f"{key}.json"
    if cf.exists():
        return json.loads(cf.read_text(encoding="utf-8"))["content"]
    content = _openai_chat(model, prompt)
    cf.write_text(json.dumps({"content": content}), encoding="utf-8")
    return content

def _parse_json_map(text):
    i, j = text.find("{"), text.rfind("}")
    if i == -1 or j == -1:
        return {}
    try:
        return json.loads(text[i:j + 1])
    except Exception:
        return {}

def _norm(s):
    return str(s).strip().strip("<>").lower()

# adversary(): recovery rate of masked identifiers for one file at one tier under one model.
# This is the hook that replaces the parametric model; real study drives it with live LLM calls.
def adversary(file_id, tier, model):
    code = masked_by_file[tier][file_id]
    idmap = id_maps.get(str(file_id), {})
    present = [p for p in idmap if p in code]
    sample = present[:K_IDENTS]
    if not sample:
        return None
    prompt = ("You are a security analyst testing whether masked source code leaks its original identifiers. "
              "Some identifiers were replaced by placeholder tokens of the form <ID_xxxxxx>. For each placeholder "
              "listed, infer the MOST LIKELY original identifier name from context. Return ONLY a JSON object "
              "mapping each placeholder to your single best guess, no explanation.\n"
              f"Placeholders: {sample}\n\nCode:\n{code}")
    guesses = _parse_json_map(_cached_chat(model, prompt))
    correct = sum(1 for p in sample if _norm(guesses.get(p, "")) == _norm(idmap[p]))
    return correct / len(sample)

masked_by_file = {t: {r["file_id"]: r["code"] for r in masked[t]} for t in TIERS}

# --- Parametric fallback (documented) --------------------------------------
PARAM_REMAINING = {"C0": 1.00, "C1": 1.00, "C2": 0.45, "C3": 0.12, "C4": 0.06}
PARAM_STRENGTH = {"weak": 0.70, "mid": 0.90, "strong": 1.00}
def parametric_risk(tier, level):
    base = PARAM_STRENGTH[level] * PARAM_REMAINING[tier]
    return base * (0.5 if not REVERSIBLE[tier] else 1.0)

# --- Compute reconstruction risk per tier x model/level, with bootstrap CIs ---
sample_files = sorted(int(fid) for fid in id_maps.keys())[:N_ADV_FILES]
risk_rows, sample_rows = [], []
for tier in TIERS:
    row = {"tier": tier, "reversible": REVERSIBLE[tier]}
    for level, model in MODELS.items():
        if tier in ("C0", "C1"):
            m, lo, hi = 1.0, 1.0, 1.0  # identifiers exposed; no reconstruction needed
        elif USE_LLM:
            vals = []
            for fid in sample_files:
                v = adversary(fid, tier, model)
                if v is not None:
                    vals.append(v)
                    sample_rows.append({"tier": tier, "model": level, "file_id": fid, "recovery": v})
            if vals:
                m = float(np.mean(vals))
                lo, hi = boot_ci(vals)
            else:
                m = parametric_risk(tier, level)
                lo = hi = m
        else:
            m = parametric_risk(tier, level)
            lo = hi = m
        row[f"risk_{level}"], row[f"risk_{level}_lo"], row[f"risk_{level}_hi"] = m, lo, hi
    row["reconstruction_risk"] = row["risk_mid"]         # primary series used downstream
    row["reconstruction_risk_lo"] = row["risk_mid_lo"]
    row["reconstruction_risk_hi"] = row["risk_mid_hi"]
    risk_rows.append(row)

risk_df = pd.DataFrame(risk_rows)
risk_df["mode"] = "llm" if USE_LLM else "parametric"
risk_df["n_adv_files"] = N_ADV_FILES
risk_df.to_csv(TAB / "risk_by_tier.csv", index=False)
if sample_rows:
    pd.DataFrame(sample_rows).to_csv(TAB / "risk_samples.csv", index=False)

# --- Figure: utility proxy across tiers ------------------------------------
fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.plot(util_df["tier"], util_df["context_retention"], color=GRAYS[0], marker=MARKERS[0],
        linewidth=1.8, label="Context retention")
ax.plot(util_df["tier"], util_df["task_success_proxy"], color=GRAYS[2], marker=MARKERS[1],
        linestyle="--", linewidth=1.8, label="Task-success proxy")
ax.set_xlabel("Masking tier")
ax.set_ylabel("Utility (retention / proxy)")
ax.set_ylim(-0.03, 1.03)
ax.legend(frameon=False, loc="lower left")
save_fig(fig, "fig_utility_by_tier")

# --- Figure: reconstruction risk by adversary tier, with 95% bootstrap CIs --
fig, ax = plt.subplots(figsize=(5.4, 3.6))
x = np.arange(len(TIERS))
for i, level in enumerate(["weak", "mid", "strong"]):
    y = risk_df[f"risk_{level}"].to_numpy()
    lo = risk_df[f"risk_{level}_lo"].to_numpy()
    hi = risk_df[f"risk_{level}_hi"].to_numpy()
    yerr = np.clip(np.vstack([y - lo, hi - y]), 0, None)
    label = f"{level} ({MODELS[level]})" if USE_LLM else f"{level} (parametric)"
    ax.errorbar(x + (i - 1) * 0.07, y, yerr=yerr, color=GRAYS[i], marker=MARKERS[i],
                linewidth=1.5, capsize=3, label=label)
ax.set_xticks(x)
ax.set_xticklabels(TIERS)
ax.set_xlabel("Masking tier")
ax.set_ylabel("Reconstruction risk")
ax.set_ylim(-0.03, 1.03)
ax.legend(frameon=False, loc="upper right", fontsize=8)
save_fig(fig, "fig_reconstruction_risk_by_tier")

print("adversary mode:", "LLM (OpenAI)" if USE_LLM else "parametric fallback (no API key)")
print("utility by tier:")
print(util_df.to_string(index=False))
print("reconstruction risk by tier:")
print(risk_df[["tier", "risk_weak", "risk_mid", "risk_strong", "reconstruction_risk", "mode"]].to_string(index=False))


adversary mode: LLM (OpenAI)
utility by tier:
tier  context_retention  task_success_proxy
  C0               1.00            0.964429
  C1               1.00            0.964429
  C2               0.55            0.645656
  C3               0.30            0.289050
  C4               0.30            0.289050
reconstruction risk by tier:
tier  risk_weak  risk_mid  risk_strong  reconstruction_risk mode
  C0   1.000000  1.000000     1.000000             1.000000  llm
  C1   1.000000  1.000000     1.000000             1.000000  llm
  C2   0.119913  0.324719     0.645346             0.324719  llm
  C3   0.121160  0.319701     0.579788             0.319701  llm
  C4   0.121160  0.319701     0.579788             0.319701  llm
